In [1]:
!pip3 install pandas numpy

In [2]:
import pandas as pd 
import numpy as np

In [3]:
def sort_to_loss_train():
    import os
    from pathlib import Path
    
    unzipped_path: Path = os.path.join(os.getcwd() + '/unzipped')
    
    if not os.path.exists(unzipped_path):
        print(f"Error: The directory '{unzipped_path}' does not exist.")
        print("You must do: create ZIP -> unzip the new samples")
        return
    srt = {"train": [], "loss": [], 'neural_structure': []} 
    
    for i,name in enumerate(os.listdir(unzipped_path)):
        match name[:1]:  
            case 'l': 
                #print(f"{i} loss")
                srt["loss"].append(os.path.join(unzipped_path, name))
            case 't':
                #print(f"{i} train")  
                srt["train"].append(os.path.join(unzipped_path, name))
            case 'N':
                srt['neural_structure'].append(os.path.join(unzipped_path, name))
            case _ as e:
                print("I had to stope the loop - something unexpected happended")
                print(f"error: {e}") 
                exit(2)
    return srt
sort_to_loss_train()

I had to stope the loop - something unexpected happended
error: .


{'train': ['/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_11.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_10.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_9.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_8.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_6.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_7.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_5.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_4.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_1.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_3.csv',
  '/Users/samuel/Documents/git/neuralparser/unzipped/trained_results_net_2.csv'],
 'loss': ['/Users/samuel/Documents/git/neuralparser/unzipped/loss_history_net_9.csv',
  '/Users/s

In [124]:
import pandas as pd
import numpy as np 
import pathlib
from pathlib import Path

csvs = sort_to_loss_train()
loss_csvs = csvs['loss']
train_csvs = csvs['train']

def to_path(df):
    if not isinstance(df,list):
        raise ValueError("Can't convert string to deref")
    
    tp = pathlib.Path(*df[:1])
    return tp

dft = pd.read_csv(to_path(train_csvs))
dfT = pd.read_csv(to_path(loss_csvs))

pd.set_option('display.max_colwidth', None)
print(f"Neural Len: {dft.shape}")
print("Info: ", dft.head(n=3))

print("loss inf: ", dfT.head(n=3))

I had to stope the loop - something unexpected happended
error: .
Neural Len: (10000, 2)
Info:     predicted_digit  confidence_score
0                0          0.494351
1                1          0.641322
2                2          0.247405
loss inf:     epoch_loss
0    2.315976
1    2.271018
2    2.230662


In [14]:
def to_path(df):
    if not isinstance(df,list):
        raise ValueError(f"Can't convert {type(df)} to deref")
    
    tp = pathlib.Path(*df[:1])
    return tp

import re
import pandas as pd

def parse_to_math(sliced):
    sequential_token = re.compile(r'Sequential\(')
    linear_token = re.compile(r'\(?\d+\)?:Linear\(in_features=(\d+),out_features=(\d+),bias=(True|False)\)')
    relu_token = re.compile(r'\(?\d+\)?:ReLU\(\)')
    
    parsed_symbols = []
    parsed_dict = {} 
    for i,line in enumerate(sliced):
        if sequential_token.search(line):
            parsed_symbols.append("SEQ[")
            
        elif linear_token.match(line):
            match = linear_token.match(line)
            in_f = match.group(1)   # Grabs 784, 512, etc.
            out_f = match.group(2)  # Grabs 512, 10, etc.
            bias = match.group(3)   # Grabs True/False
             
            parsed_symbols.append(f"L({in_f},{out_f},b={bias})")
            #NO BIAS (bias) 
            parsed_dict[f"L{i}"] = [in_f, out_f]
            
        elif relu_token.match(line):
            parsed_symbols.append(f"R{i}()")
            parsed_dict[f"R{i}"] = 0
            
        elif line == ')':
            parsed_symbols.append("]")

    # 3. Join the processed symbols into a readable stream path
    #print(parsed_symbols) 
    return parsed_dict

def read_and_parse(): 
    f = sort_to_loss_train()['neural_structure']
    o = pd.read_csv(to_path(f))
    parsed = {} 
    for i in range(0, o['architecture'].size):
        sliced = o['architecture'][i].replace(' ', '').split('\n')
        parsed[i] = parse_to_math(sliced)
    return parsed
    
    #print(parse_to_math(o[0]))

read_and_parse()

I had to stope the loop - something unexpected happended
error: .


{0: {'L1': ['784', '512'], 'R2': 0, 'L3': ['512', '10']},
 1: {'L1': ['784', '256'], 'R2': 0, 'L3': ['256', '10']},
 2: {'L1': ['784', '128'], 'R2': 0, 'L3': ['128', '10']},
 3: {'L1': ['784', '512'],
  'R2': 0,
  'L3': ['512', '256'],
  'R4': 0,
  'L5': ['256', '10']},
 4: {'L1': ['784', '256'],
  'R2': 0,
  'L3': ['256', '128'],
  'R4': 0,
  'L5': ['128', '10']},
 5: {'L1': ['784', '512'],
  'R2': 0,
  'L3': ['512', '128'],
  'R4': 0,
  'L5': ['128', '10']},
 6: {'L1': ['784', '512'],
  'R2': 0,
  'L3': ['512', '256'],
  'R4': 0,
  'L5': ['256', '128'],
  'R6': 0,
  'L7': ['128', '10']},
 7: {'L1': ['784', '256'],
  'R2': 0,
  'L3': ['256', '256'],
  'R4': 0,
  'L5': ['256', '256'],
  'R6': 0,
  'L7': ['256', '10']},
 8: {'L1': ['784', '128'],
  'R2': 0,
  'L3': ['128', '64'],
  'R4': 0,
  'L5': ['64', '10']},
 9: {'L1': ['784', '256'],
  'R2': 0,
  'L3': ['256', '64'],
  'R4': 0,
  'L5': ['64', '10']},
 10: {'L1': ['784', '512'],
  'R2': 0,
  'L3': ['512', '64'],
  'R4': 0,
  'L5': 

In [6]:
#$$\text{Estimated Accuracy} = (784 \times 512 \times W_1) + (512 \times B_1) + R \times ((512 \times 10 \times W_2) + (10 \times B_2))$$

In [122]:
def variable_parse(biases:list, weights:list):
    math = read_and_parse()
    
    equation = []
    equation_counter = 0
    opening_bracket = False
    equations_parsed = {}
    closing_bracket_relu = False #for parsing
    
    for EI in range(0,len(math)):
        for i,(k,v) in enumerate(math[EI].items()):
            if k.startswith("L"):
                equation.append(f"({v[0]} * {v[1]} * {weights[equation_counter]}) + ({v[1]} * {biases[equation_counter]}) ")
                equation_counter += 1
                if opening_bracket:
                    if opening_bracket and i + 1 == len(math[EI].keys()):
                        equation.append(")")
                        opening_bracket = False
                    equation.append(")")
                    opening_bracket = False
                
                if closing_bracket_relu:
                    equation.append("")
                    closing_bracket_relu = False
            
            if k.startswith("R"):
                equation.append(f" + 0.5 * ( ") #relu = 0.5
                closing_bracket_relu = True
                opening_bracket = True
            
            if i + 1 == len(math[EI].keys()):
                equation.append("") 
            
        joined = ''.join(equation).replace(' ', '')
        joined = joined.replace('++','+')
        joined = joined.replace(")))", "))")
        equations_parsed[EI] = joined
        equation = []
        joined = ''
        equation_counter = 0
    return equations_parsed


def count_linears():
    math = read_and_parse()
    max_symbols = 0
    tmp_symbols = 0
    for i in range(0, len(math.keys())):
        for k in math[i].keys():
            if k.startswith("L"):
                tmp_symbols += 1
        
        if tmp_symbols > max_symbols:
            max_symbols = tmp_symbols
        tmp_symbols = 0
    return max_symbols

#UPDATING BIASES + WEIGHTS HAPPENS HERE vvvvvvvvvv

In [154]:
def calc_f_acc():
    import pandas as pd
    import numpy as np
    import pathlib
    from pathlib import Path
    
    def go_to_loop():
        test_res = pd.read_csv(Path('/Users/samuel/Documents/git/neuralparser/fashion-mnist_test.csv'))
        resutls = []
        csvs = sort_to_loss_train()
        for df_i in range(0, len(csvs['loss'])):
            loss_csvs = csvs['train'][df_i]
            loss_csv_0 = pd.read_csv(loss_csvs) 
            correct = 0
            missed = 0
            
            for i in range(0, len(test_res['label'])):
                if test_res['label'][i] == loss_csv_0['predicted_digit'][i]:
                    correct += 1
                else:
                    missed += 1
            
            ratio = lambda correct, size: correct / size 
            #print(f"Correct: {correct} | Missed: {missed} | Ratio {ratio(correct=correct, size=test_res['label'].size)}")
            resutls.append(ratio(correct=correct, size=test_res['label'].size))
        
        return resutls
    
    out = go_to_loop()
    return out
print(len(calc_f_acc()))

I had to stope the loop - something unexpected happended
error: .
11


In [ ]:
OUT = variable_parse(np.linspace(0.9,1,count_linears()), np.linspace(0.9,1,count_linears()))
accuracies = calc_f_acc()
accuracies_size = len(accuracies)

def assemble_stats():
    stats = {} 
    for i in range(0, len(OUT.keys())):
        stats[f"net_{i}"] = {"equation":OUT[i],"equation_eval": eval(OUT[i]), "equation_accuracy": accuracies[i]}
    return stats

#we need to send update to those equations (USE OLD PARSER - should be easy..)
print(assemble_stats())

I had to stope the loop - something unexpected happended
error: .
I had to stope the loop - something unexpected happended
error: .
I had to stope the loop - something unexpected happended
error: .
I had to stope the loop - something unexpected happended
error: .
{'net_0': {'equation': '(784*512*0.9)+(512*0.9)+0.5*((512*10*0.9333333333333333)+(10*0.9333333333333333))', 'equation_eval': 364122.0, 'equation_accuracy': 0.6434}, 'net_1': {'equation': '(784*256*0.9)+(256*0.9)+0.5*((256*10*0.9333333333333333)+(10*0.9333333333333333))', 'equation_eval': 182063.33333333334, 'equation_accuracy': 0.6242}, 'net_2': {'equation': '(784*128*0.9)+(128*0.9)+0.5*((128*10*0.9333333333333333)+(10*0.9333333333333333))', 'equation_eval': 91034.0, 'equation_accuracy': 0.5289}, 'net_3': {'equation': '(784*512*0.9)+(512*0.9)+0.5*((512*256*0.9333333333333333)+(256*0.9333333333333333))+0.5*((256*10*0.9666666666666667)+(10*0.9666666666666667))', 'equation_eval': 424256.5666666667, 'equation_accuracy': 0.6375}, '